In [ ]:
import torch

print(f"torch version: {torch.__version__}")
print(f"torch cuda version: {torch.version.cuda}")
device = torch.device(torch.cuda.current_device() if torch.cuda.is_available() else "cpu")
print(f"torch device: {device}")
print(f"device name: {torch.cuda.get_device_name(device) if torch.cuda.is_available() else 'cpu'}")

torch version: 2.6.0+cu126
torch cuda version: 12.6
torch device: cuda:0
device name: NVIDIA GeForce RTX 4090 Laptop GPU


In [ ]:
import json
import logging
import os
from enum import EnumType

import datasets
import pandas as pd
import tqdm
from datasets import load_dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
)
from trl import SFTTrainer

EXIT_FAILURE = 1
EXIT_SUCCESS = 0

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger(__name__)


class MissionType(EnumType):
    WA = 1
    SA = 2
    SE = 3


MISC_DATA_PATH = os.path.join(os.path.pardir, "MISC")
JOB_DATA_PATH = os.path.join(MISC_DATA_PATH, "job_data_files")


class JobDataPath(EnumType):
    WA_DEV = "work_arrangements_development_set.csv"
    WA_TEST = "work_arrangements_test_set.csv"
    SA_DEV = "salary_labelled_development_set.csv"
    SA_TEST = "salary_labelled_test_set.csv"
    SE_DEV = "seniority_labelled_development_set.csv"
    SE_TEST = "seniority_labelled_test_set.csv"


class JobData:
    def __init__(self, mission_type: MissionType):
        func_join_path = lambda x: os.path.join(JOB_DATA_PATH, x)
        match mission_type:
            case MissionType.WA:
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.WA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.WA_TEST))
            case MissionType.SA:
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SA_TEST))
            case MissionType.SE:
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SE_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SE_TEST))
            case _:
                self.df_dev = None
                self.df_test = None
                raise ValueError(f"Invalid mission type: {mission_type}")

In [ ]:
class MistralData:
    def __init__(self, mission_type: MissionType):
        func_join_path = lambda x: os.path.join(JOB_DATA_PATH, x)
        match mission_type:
            case MissionType.WA:
                # Load the data
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.WA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.WA_TEST))
                # Create train dataset
                train_jsonl_list = self.__output_prompt_jsonl_WA(self.df_dev)
                train_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "work_arrangements_train.jsonl"
                )
                self.__write_jsonl(train_jsonl_path, train_jsonl_list)
                self.train_dataset = load_dataset("json", data_files=train_jsonl_path)
                # Create test dataset
                test_jsonl_list = self.__output_prompt_jsonl_WA(self.df_test)
                test_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "work_arrangements_test.jsonl"
                )
                self.__write_jsonl(test_jsonl_path, test_jsonl_list)
                self.test_dataset = load_dataset("json", data_files=test_jsonl_path)
            case MissionType.SA:
                # Load the data
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SA_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SA_TEST))
                # Create train dataset
                train_jsonl_list = self.__output_prompt_jsonl_SA(self.df_dev)
                train_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "salary_train.jsonl"
                )
                self.__write_jsonl(train_jsonl_path, train_jsonl_list)
                self.train_dataset = load_dataset("json", data_files=train_jsonl_path)
                # Create test dataset
                test_jsonl_list = self.__output_prompt_jsonl_SA(self.df_test)
                test_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "salary_test.jsonl"
                )
                self.__write_jsonl(test_jsonl_path, test_jsonl_list)
                self.test_dataset = load_dataset("json", data_files=test_jsonl_path)
            case MissionType.SE:
                # Load the data
                self.df_dev = pd.read_csv(func_join_path(JobDataPath.SE_DEV))
                self.df_test = pd.read_csv(func_join_path(JobDataPath.SE_TEST))
                # Create train dataset
                train_jsonl_list = self.__output_prompt_jsonl_SE(self.df_dev)
                train_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "seniority_train.jsonl"
                )
                self.__write_jsonl(train_jsonl_path, train_jsonl_list)
                self.train_dataset = load_dataset("json", data_files=train_jsonl_path)
                # Create test dataset
                test_jsonl_list = self.__output_prompt_jsonl_SE(self.df_test)
                test_jsonl_path = os.path.join(
                    MISC_DATA_PATH, "mistral", "seniority_test.jsonl"
                )
                self.__write_jsonl(test_jsonl_path, test_jsonl_list)
                self.test_dataset = load_dataset("json", data_files=test_jsonl_path)
            case _:
                self.df_dev = None
                self.df_test = None
                raise ValueError(f"Invalid mission type: {mission_type}")
        pass

    def __output_prompt_jsonl_SA(self, df: pd.DataFrame) -> list:
        logger.info(f"Creating jsonl for {df.keys()} rows")
        jsonl_list = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Creating jsonl"):
            jsonl_list.append(
                {
                    "instruction": (
                        "You are a helpful assistant."
                        "Your task is to classify the salary level of the job advertisement. "
                        "The format will be [NUMBER]-[NUMBER]-[CURRENCY SYMBOL]-[HOURLY|DAILY|MONTHLY|YEARLY]."
                    ),
                    "input": (  # job_title job_ad_details nation_short_desc salary_additional_text
                        f"The job title is: {row['job_title']}."
                        f"Further details: {row['job_ad_details']}."
                        f"Country Codes: {row['nation_short_desc']}."
                        f"Salary information: {row['salary_additional_text']}."
                    ),
                    "output": f"Answer: {row['y_true']}",
                }
            )
        return jsonl_list

    def __output_prompt_jsonl_SE(self, df: pd.DataFrame) -> list:
        logger.info(f"Creating jsonl for {df.keys()} rows")
        jsonl_list = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Creating jsonl"):
            jsonl_list.append(
                {
                    "instruction": (
                        "You are a helpful assistant."
                        "Your task is to classify the seniority level of the job advertisement."
                        "The format will be [SENIORITY LEVEL]."
                    ),
                    "input": (
                        f"The job title is: {row['job_title']}."
                        f"To summarize the job: {row['job_summary']}."
                        f"Further details: {row['job_ad_details']}"
                        f"The classification is: {row["classification_name"]} / {row["subclassification_name"]}."
                    ),
                    "output": f"Answer: {row['y_true']}",
                }
            )
        return jsonl_list

    def __output_prompt_jsonl_WA(self, df: pd.DataFrame) -> list:
        logger.info(f"Creating jsonl for {df.keys()} rows")
        jsonl_list = []
        for _, row in tqdm.tqdm(df.iterrows(), total=len(df), desc="Creating jsonl"):
            jsonl_list.append(
                {
                    "instruction": (
                        "You are a helpful assistant."
                        "Your task is to classify which type of work arrangements the job advertisement belongs to."
                        "There are three types: ['OnSite', 'Remote', 'Hybrid']."
                    ),
                    "input": row["job_ad"],
                    "output": f"Answer: {row['y_true']}",
                }
            )
        return jsonl_list

    def __write_jsonl(self, jsonl_path: str, data: list) -> None:
        # if file exists, remove it
        if os.path.exists(jsonl_path):
            os.remove(jsonl_path)
        # create file and write data
        with open(jsonl_path, "w", encoding="utf-8") as f:
            for item in data:
                f.write(json.dumps(item, ensure_ascii=False) + "\n")

    def format_prompt(self, example) -> str:
        return f"<s>[INST] {example["instruction"]} {example["input"]} [/INST] {example["output"]} </s>"

    def get_map_data(self, dataset: datasets.Dataset) -> datasets.Dataset:
        return dataset.map(
            lambda example: {"text": self.format_prompt(example)},
            remove_columns=["instruction", "input", "output"],
        )


print("Loading Mistral data...")
md = MistralData(MissionType.WA)
md.df_dev.head()

2025-04-09 00:52:34 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows


Loading Mistral data...


Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 49521.30it/s]


Generating train split: 0 examples [00:00, ? examples/s]

2025-04-09 00:52:35 - INFO - Creating jsonl for Index(['id', 'job_ad', 'y_true'], dtype='object') rows
Creating jsonl: 100%|██████████| 99/99 [00:00<00:00, 48508.89it/s]


Generating train split: 0 examples [00:00, ? examples/s]

,id,job_ad,y_true
0,79484040,Job title: CEO\nAbstract: Exciting opportunity...,Remote
1,80331384,Job title: Home-Based Online ESL Teacher (Onli...,Remote
2,79721069,"Job title: Safeguarding, De La Salle\nAbstract...",Hybrid
3,80190376,Job title: Delivery Driver\nAbstract: Pickup t...,OnSite
4,80082230,Job title: Store Supervisor\nAbstract: We are ...,OnSite


In [ ]:
class MistralConfig:
    def __init__(self, mission_type: MissionType) -> None:
        self.mission_type = mission_type

        self.bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_use_double_quant=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16,
        )
        self.lora_config = LoraConfig(
            r=8,
            lora_alpha=16,
            lora_dropout=0.1,
            bias="none",
            task_type="CAUSAL_LM",
            target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj"],
        )


class MistralModel:
    def __init__(self, MistralConfig: MistralConfig, MistralData: MistralData) -> None:
        self.model_id = "mistralai/Mistral-7B-Instruct-v0.3"

        self.device = torch.device(
            torch.cuda.current_device() if torch.cuda.is_available() else "cpu"
        )

        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_id,
            quantization_config=MistralConfig.bnb_config,
            torch_dtype=torch.bfloat16,
            device_map="auto",
        )

        self.model.config.use_cache = False
        self.model.config.pretraining_tp = 1

        self.tokenizer = AutoTokenizer.from_pretrained(self.model_id)
        self.tokenizer.pad_token = self.tokenizer.eos_token

        self.model = prepare_model_for_kbit_training(self.model)
        self.model = get_peft_model(self.model, MistralConfig.lora_config)
        self.model.generation_config.pad_token_id = self.tokenizer.pad_token_id
        self.model.gradient_checkpointing_enable()

        self.fine_tuing_data = MistralData.get_map_data(MistralData.train_dataset)

        logger.info("Model and tokenizer loaded successfully.")

    def build_train_arguments(self) -> TrainingArguments:
        return TrainingArguments(
            output_dir="./results",
            num_train_epochs=3,
            per_device_train_batch_size=2,
            gradient_accumulation_steps=4,
            optim="paged_adamw_8bit",
            save_steps=5,
            logging_steps=1,
            learning_rate=2e-4,
            weight_decay=0.001,
            fp16=False,
            bf16=False,
            max_grad_norm=0.3,
            max_steps=-1,
            warmup_ratio=0.03,
            group_by_length=True,
            lr_scheduler_type="linear",
            report_to="wandb",
            seed=42,
        )

    def train(self, MistralConfig: MistralConfig) -> None:
        trainer_args = self.build_train_arguments()
        trainer = SFTTrainer(
            model=self.model,
            train_dataset=self.fine_tuing_data["train"],
            args=trainer_args,
            peft_config=MistralConfig.lora_config,
        )
        trainer.train()

        # Save the model
        match MistralConfig.mission_type:
            case MissionType.WA:
                trainer.save_model("./mistral-7b-lora-WA")
                self.tokenizer.save_pretrained("./mistral-7b-lora-WA")
            case MissionType.SA:
                trainer.save_model("./mistral-7b-lora-SA")
                self.tokenizer.save_pretrained("./mistral-7b-lora-SA")
            case MissionType.SE:
                trainer.save_model("./mistral-7b-lora-SE")
                self.tokenizer.save_pretrained("./mistral-7b-lora-SE")
            case _:
                raise ValueError(f"Invalid mission type: {MistralConfig.mission_type}")

        logger.info("Model trained and saved successfully.")

In [ ]:
mc = MistralConfig(MissionType.WA)
mm = MistralModel(mc, md)

logger.info("Training the model...")

mm.train(mc)

logger.info("Model training completed.")

2025-04-09 00:52:43 - INFO - We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

Map:   0%|          | 0/99 [00:00<?, ? examples/s]

2025-04-09 00:52:52 - INFO - Model and tokenizer loaded successfully.
2025-04-09 00:52:52 - INFO - Training the model...


Converting train dataset to ChatML:   0%|          | 0/99 [00:00<?, ? examples/s]

...

Tokenizing train dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/99 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: mingyuancui (mingyuancui-unsw) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Step,Training Loss
1,2.403600
2,2.624500
3,2.550100
4,2.073300
5,2.107600
6,2.223900
7,1.960800
8,1.949400
9,1.764800
10,1.844000


2025-04-09 01:05:01 - INFO - Model trained and saved successfully.
2025-04-09 01:05:01 - INFO - Model training completed.


In [13]:
import re


def find_answer(text: str, mission_type: MissionType = MissionType.WA) -> str:
    regex = r"Answer: (.*)"
    result = re.search(regex, text)
    if result:
        return result.group(1).strip()
    else:
        return "None"

# find_answer = lambda text: "".join(reversed(text.split(" ", 1)[0]))

def predict(MistralData: MistralData) -> list[str]:
    y_pred = []
    test_data = MistralData.test_dataset["train"]
    logger.info("Predicting...")
    for i in tqdm.tqdm(range(len(test_data))):
        input_text = test_data["input"][i]
        instruction_text = test_data["instruction"][i]
        format_input = f"<s>[INST] {instruction_text} {input_text} [/INST]"
        input_ids = mm.tokenizer(format_input, return_tensors="pt").input_ids.to(mm.device)
        attention_mask = mm.tokenizer(format_input, return_tensors="pt").attention_mask.to(mm.device)
        mm.model.gradient_checkpointing_enable()
        mm.model.generation_config.pad_token_id = mm.tokenizer.pad_token_id
        output = mm.model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            num_return_sequences=1
        )
        mm.model.gradient_checkpointing_enable()
        # print(sequences[0]["generated_text"])
        output_text = mm.tokenizer.decode(output[0], skip_special_tokens=True)
        # print(f"Output: {output_text}")
        a = find_answer(output_text)
        # print(f"Answer: {a}")
        y_pred.append(a)
    return y_pred


y_pred = predict(md)
y_test = md.df_test["y_true"].tolist()

print(f"y_pred: {y_pred}")
print(f"y_test: {y_test}")

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print(classification_report(y_test, y_pred, digits=4))
print(f"F1 Score: {f1_score(y_test, y_pred, average='weighted')}")

2025-04-09 01:24:36 - INFO - Predicting...
100%|██████████| 99/99 [02:26<00:00,  1.48s/it]


y_pred: ['OnSite', 'OnSite', 'Remote', 'OnSite', 'Remote', 'OnSite', 'Remote', 'Remote', 'OnSite', 'Remote', 'Hybrid', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Remote', 'Remote', 'OnSite', 'Remote', 'Hybrid', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'Remote', 'Remote', 'OnSite', 'Remote', 'OnSite', 'Remote', 'Remote', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Remote', 'Hybrid', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'Hybrid', 'Remote', 'Remote', 'Remote', 'Hybrid', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Remote', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'Hybrid', 'OnSite', 'Hybrid', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'Remote', 'OnSite', 'Remote', 'OnSite', 'OnSite', 'OnSite']
y